## 深度域 Auto Well Tie 子波提取

本 notebook 使用与确定性子波提取相同的数据准备流程：从 TVDSS 深度域地震提取井旁道，用井速构建局部相对时深表，再运行 `autotie.tie_v1`。最后将 auto well tie 输出的长子波以 0 ms 为中心裁剪到目标 201 ms，并做 L2 能量归一化。


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_well_heads_petrel, load_vp_rho_logset_from_las
from cup.seismic.survey import open_survey
from wtie.modeling.modeling import ConvModeler
from wtie.optimize import autotie
from wtie.processing import grid
from wtie.processing.logs import interpolate_nans
from wtie.utils import viz
from wtie.utils.datasets import tutorial
from wtie.utils.datasets.utils import InputSet

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 30)

In [ ]:
well_name = "NW11"

data_root = repo_root / "data"
las_file = data_root / "vertical_well_las_target_qyz" / f"{well_name}.las"
well_heads_file = data_root / "raw" / "well_heads"
seismic_file = data_root / "raw" / "mero_84_coord_extend"
tutorial_folder = data_root / "tutorial"
model_state_dict = tutorial_folder / "trained_net_state_dict.pt"
network_parameters_file = tutorial_folder / "network_parameters.yaml"

output_dir = data_root / "output_vertical_well_auto_tie_depth_20260428"
output_dir.mkdir(parents=True, exist_ok=True)

target_crop_ms = 201.0

search_space_config = dict(
    logs_median_size_values=[51, 71, 91, 111],  # 5.1-11.1 m
    logs_median_threshold_bounds=[0.5, 3.0],
    logs_std_bounds=[20, 50],  # sigma = 2-5 m
    table_t_shift_bounds=[-0.030, 0.030],
)

segy_options = {
    "iline": 5,
    "xline": 21,
    "istep": 1,
    "xstep": 4,
}

for path in (las_file, well_heads_file, seismic_file, model_state_dict, network_parameters_file):
    assert path.exists(), f"Missing input file: {path}"

print("LAS:", las_file)
print("Seismic:", seismic_file)
print("Output dir:", output_dir)


## 工具函数


In [ ]:
def to_jsonable(value):
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return to_jsonable(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value


def build_auto_tie_search_space(config: dict) -> list[dict]:
    return [
        dict(
            name="logs_median_size",
            type="choice",
            values=config["logs_median_size_values"],
            value_type="int",
            is_ordered=True,
            sort_values=True,
        ),
        dict(
            name="logs_median_threshold",
            type="range",
            bounds=config["logs_median_threshold_bounds"],
            value_type="float",
        ),
        dict(
            name="logs_std",
            type="range",
            bounds=config["logs_std_bounds"],
            value_type="float",
        ),
        dict(
            name="table_t_shift",
            type="range",
            bounds=config["table_t_shift_bounds"],
            value_type="float",
        ),
    ]


def depth_trace_to_twt(depth_tvdss: np.ndarray, amp: np.ndarray, tdt_df: pd.DataFrame, dt_s: float) -> grid.Seismic:
    depth_tvdss = np.asarray(depth_tvdss, dtype=float)
    amp = interpolate_nans(amp, method="linear")
    z_tdt = tdt_df["tvdss_m"].to_numpy(dtype=float)
    t_tdt = tdt_df["twt_s"].to_numpy(dtype=float)
    z_min = max(float(depth_tvdss.min()), float(z_tdt[0]))
    z_max = min(float(depth_tvdss.max()), float(z_tdt[-1]))
    if z_max <= z_min:
        raise ValueError("Depth trace and local TDT do not overlap.")
    dz = float(np.median(np.diff(z_tdt)))
    z_regular = np.arange(z_min, z_max + 0.5 * dz, dz)
    amp_z = np.interp(z_regular, depth_tvdss, amp)
    twt_z = np.interp(z_regular, z_tdt, t_tdt)
    twt_regular = np.arange(twt_z[0], twt_z[-1] + 0.5 * dt_s, dt_s)
    amp_t = np.interp(twt_regular, twt_z, amp_z)
    return grid.Seismic(amp_t, twt_regular, "twt", name="Depth-derived seismic")


def clip_logset_by_md(logset: grid.LogSet, md_min: float, md_max: float) -> grid.LogSet:
    mask = (logset.basis >= md_min) & (logset.basis <= md_max)
    if np.count_nonzero(mask) < 10:
        raise ValueError("Too few log samples in requested MD window.")
    logs = {}
    for key, log in logset.Logs.items():
        logs[key] = grid.Log(
            log.values[mask], log.basis[mask], "md", name=log.name, unit=log.unit, allow_nan=log.allow_nan
        )
    return grid.LogSet(logs)


def crop_wavelet_center_energy_normalize(wavelet: grid.Wavelet, target_ms: float) -> tuple[grid.Wavelet, dict]:
    dt = float(wavelet.sampling_rate)
    target_s = target_ms / 1000.0
    n_target = int(round(target_s / dt))
    if n_target % 2 == 0:
        n_target += 1
    n_target = min(n_target, wavelet.size)
    if n_target % 2 == 0:
        n_target -= 1
    center_idx = int(np.argmin(np.abs(wavelet.basis)))
    half = n_target // 2
    start = max(0, center_idx - half)
    end = start + n_target
    if end > wavelet.size:
        end = wavelet.size
        start = end - n_target
    values = np.asarray(wavelet.values[start:end], dtype=float).copy()
    basis = np.asarray(wavelet.basis[start:end], dtype=float).copy()
    energy = float(np.sqrt(np.sum(values**2)))
    if not np.isfinite(energy) or energy <= 0:
        raise ValueError("Cannot energy-normalize a zero-energy wavelet.")
    values_norm = values / energy
    cropped = grid.Wavelet(values_norm, basis, name="Auto well tie wavelet cropped 201ms energy-normalized")
    info = {
        "target_ms": target_ms,
        "dt_s": dt,
        "original_samples": int(wavelet.size),
        "cropped_samples": int(cropped.size),
        "cropped_span_ms": float((basis[-1] - basis[0]) * 1000.0),
        "cropped_nominal_ms": float(cropped.size * dt * 1000.0),
        "pre_normalization_l2_energy": energy,
        "post_normalization_l2_energy": float(np.sqrt(np.sum(values_norm**2))),
    }
    return cropped, info


def scaled_synthetic_for_qc(
    modeler: ConvModeler, wavelet: grid.Wavelet, reflectivity: grid.Reflectivity, seismic: grid.Seismic
) -> tuple[np.ndarray, float, float, float]:
    synthetic_raw = modeler(wavelet.values, reflectivity.values)
    seis = np.asarray(seismic.values, dtype=float)
    seis = seis - np.mean(seis)
    seis_std = np.std(seis)
    if seis_std <= 0:
        raise ValueError("Seismic trace has zero standard deviation.")
    seis_norm = seis / seis_std
    scale = float(np.dot(seis_norm, synthetic_raw) / max(np.dot(synthetic_raw, synthetic_raw), 1e-12))
    synthetic = scale * synthetic_raw
    corr = float(np.corrcoef(seis_norm, synthetic)[0, 1]) if np.std(synthetic) > 0 else np.nan
    nmae = float(np.sum(np.abs(seis_norm - synthetic)) / np.sum(np.abs(seis_norm)))
    return synthetic, scale, corr, nmae


def savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved", path)


## 1) 读取井数据、网络和深度域井旁道


In [ ]:
well_heads_df = import_well_heads_petrel(well_heads_file)
well_row = well_heads_df.loc[well_heads_df["Name"] == well_name]
assert not well_row.empty, f"well_name={well_name} not found in well_heads"
assert well_row.shape[0] == 1, f"well_name={well_name} duplicated in well_heads: {well_row.shape[0]}"
well_row = well_row.iloc[0]
kb_m = float(well_row["Well datum value"])
well_x = float(well_row["Surface X"])
well_y = float(well_row["Surface Y"])

logset_md_full = load_vp_rho_logset_from_las(
    las_file,
    vp_unit="us/m",
    rho_unit="g/cm3",
)

with open(network_parameters_file, "r", encoding="utf-8") as f:
    training_parameters = yaml.load(f, Loader=yaml.Loader)
wavelet_extractor = tutorial.load_wavelet_extractor(training_parameters, model_state_dict)
modeler = ConvModeler()
dt_s = float(wavelet_extractor.expected_sampling)

survey = open_survey(seismic_file, seismic_type="segy", segy_options=segy_options)
geometry_depth = survey.query_geometry(domain="depth")
il_float, xl_float = survey.coord_to_line(well_x, well_y)
assert geometry_depth["inline_min"] <= il_float <= geometry_depth["inline_max"]
assert geometry_depth["xline_min"] <= xl_float <= geometry_depth["xline_max"]

seismic_depth_trace = survey.import_seismic_at_well(well_x=well_x, well_y=well_y, domain="depth")
seis_depth = seismic_depth_trace.basis.astype(float)
seis_amp = interpolate_nans(seismic_depth_trace.values, method="linear")

print(f"Well XY=({well_x:.2f}, {well_y:.2f}), KB={kb_m:.2f} m")
print(f"Network expected dt={dt_s * 1000:.1f} ms")
print(f"Depth trace: z=[{seis_depth[0]:.2f}, {seis_depth[-1]:.2f}] m, dz={seismic_depth_trace.sampling_rate:.3f} m")


## 2) 共同深度窗和 Auto Tie 输入


In [ ]:
md_full = logset_md_full.basis.astype(float)
tvdss_full = md_full - kb_m
vp_full = interpolate_nans(logset_md_full.Vp.values, method="linear")

overlap_z_min = max(float(tvdss_full.min()), float(seis_depth.min()))
overlap_z_max = min(float(tvdss_full.max()), float(seis_depth.max()))
assert overlap_z_max > overlap_z_min, "Well TVDSS and seismic depth trace do not overlap."
overlap_md_min = overlap_z_min + kb_m
overlap_md_max = overlap_z_max + kb_m

logset_md = clip_logset_by_md(logset_md_full, overlap_md_min, overlap_md_max)
md_win = logset_md.basis.astype(float)
tvdss_win = md_win - kb_m
tdt_df = grid.build_local_tdt_from_vp(
    tvdss_m=tvdss_win,
    vp_mps=interpolate_nans(logset_md.Vp.values, method="linear"),
    md_m=md_win,
)
local_tdt_md = grid.TimeDepthTable(
    twt=tdt_df["twt_s"].to_numpy(),
    md=tdt_df["md_m"].to_numpy(),
)
seismic_twt = depth_trace_to_twt(seis_depth, seis_amp, tdt_df, dt_s)

inputs = InputSet(
    logset_md=logset_md,
    seismic=seismic_twt,
    table=local_tdt_md,
    wellpath=None,  # type: ignore
)

local_tdt_path = output_dir / f"local_tdt_md_{well_name}.csv"
tdt_df.to_csv(local_tdt_path, index=False)
seismic_twt_path = output_dir / f"seismic_twt_from_depth_{well_name}.csv"
pd.DataFrame({"twt_s": seismic_twt.basis, "seismic": seismic_twt.values}).to_csv(seismic_twt_path, index=False)

print(f"Overlap TVDSS window: {overlap_z_min:.2f} - {overlap_z_max:.2f} m")
print(f"Overlap MD window: {overlap_md_min:.2f} - {overlap_md_max:.2f} m")
print(f"Seismic TWT window: {seismic_twt.basis[0]:.4f} - {seismic_twt.basis[-1]:.4f} s, n={seismic_twt.size}")
print("Saved", local_tdt_path)
print("Saved", seismic_twt_path)

## 3) 运行 `autotie.tie_v1`


In [ ]:
search_params = dict(
    num_iters=60,
    similarity_std=0.02,
    suppress_runtime_warnings=True,
    show_all_warnings=False,
)
wavelet_scaling_params = dict(
    wavelet_min_scale=50000,
    wavelet_max_scale=500000,
    num_iters=60,
)
search_space = build_auto_tie_search_space(search_space_config)

outputs = autotie.tie_v1(
    inputs,
    wavelet_extractor,
    modeler,
    wavelet_scaling_params,
    search_params=search_params,
    search_space=search_space,  # type: ignore
    stretch_and_squeeze_params=None,  # type: ignore
)

raw_wavelet = outputs.wavelet
raw_wavelet_path = output_dir / f"auto_well_tie_wavelet_raw_{well_name}.csv"
pd.DataFrame({"time_s": raw_wavelet.basis, "amplitude": raw_wavelet.values}).to_csv(raw_wavelet_path, index=False)

raw_tie_path = output_dir / f"auto_well_tie_synthetic_qc_raw_{well_name}.csv"
pd.DataFrame(
    {
        "twt_s": outputs.seismic.basis,
        "seismic": outputs.seismic.values,
        "reflectivity": outputs.r.values,
        "synthetic_raw_wavelet": outputs.synth_seismic.values,
    }
).to_csv(raw_tie_path, index=False)

print(
    f"Raw auto-tie wavelet samples={raw_wavelet.size}, dt={raw_wavelet.sampling_rate * 1000:.1f} ms, span={(raw_wavelet.basis[-1] - raw_wavelet.basis[0]) * 1000:.1f} ms"  # type: ignore
)
print("Saved", raw_wavelet_path)
print("Saved", raw_tie_path)


## 4) 裁剪到 201 ms 并做能量归一化


In [ ]:
cropped_wavelet, crop_info = crop_wavelet_center_energy_normalize(raw_wavelet, target_crop_ms)  # type: ignore
crop_window_label = f"{target_crop_ms:g}ms"
cropped_wavelet_path = output_dir / f"wavelet_{crop_window_label}_{well_name}.csv"
pd.DataFrame({"time_s": cropped_wavelet.basis, "amplitude": cropped_wavelet.values}).to_csv(
    cropped_wavelet_path, index=False
)

cropped_synthetic, cropped_scale, cropped_corr, cropped_nmae = scaled_synthetic_for_qc(
    modeler=modeler,
    wavelet=cropped_wavelet,
    reflectivity=outputs.r,  # type: ignore
    seismic=outputs.seismic,  # type: ignore
)

seis_norm = outputs.seismic.values - np.mean(outputs.seismic.values)
seis_norm = seis_norm / np.std(seis_norm)
cropped_qc_path = output_dir / f"auto_well_tie_synthetic_qc_cropped_201ms_{well_name}.csv"
pd.DataFrame(
    {
        "twt_s": outputs.seismic.basis,
        "seismic_norm": seis_norm,
        "reflectivity": outputs.r.values,
        "synthetic_cropped_scaled": cropped_synthetic,
        "residual": seis_norm - cropped_synthetic,
    }
).to_csv(cropped_qc_path, index=False)

best_parameters, best_values = outputs.ax_client.get_best_parameters()  # type: ignore
best_objective_means, best_objective_covariances = best_values  # type: ignore

summary = {
    "well_name": well_name,
    "kb_m": kb_m,
    "overlap_tvdss_min_m": overlap_z_min,
    "overlap_tvdss_max_m": overlap_z_max,
    "auto_tie_search_params": search_params,
    "auto_tie_search_space_config": to_jsonable(search_space_config),
    "wavelet_scaling_params": wavelet_scaling_params,
    "auto_tie_best_parameters": to_jsonable(best_parameters),
    "auto_tie_best_objective_means": to_jsonable(best_objective_means),
    "auto_tie_best_objective_covariances": to_jsonable(best_objective_covariances),
    "raw_wavelet_samples": int(raw_wavelet.size),  # type: ignore
    "raw_wavelet_dt_s": float(raw_wavelet.sampling_rate),
    "raw_wavelet_span_ms": float((raw_wavelet.basis[-1] - raw_wavelet.basis[0]) * 1000.0),
    "crop_info": crop_info,
    "cropped_synthetic_scale": cropped_scale,
    "cropped_synthetic_corr": cropped_corr,
    "cropped_synthetic_nmae": cropped_nmae,
}
summary_path = output_dir / f"run_summary_auto_well_tie_{well_name}.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(crop_info)
print(f"Cropped synthetic: corr={cropped_corr:.3f}, nmae={cropped_nmae:.3f}, scale={cropped_scale:.3g}")
print("Saved", cropped_wavelet_path)
print("Saved", cropped_qc_path)
print("Saved", summary_path)


## 5) QC 图件


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=False)
axes[0].plot(logset_md.Vp.values, logset_md.basis - kb_m, lw=0.8)
axes[0].invert_yaxis()
axes[0].set_xlabel("Vp (m/s)")
axes[0].set_ylabel("TVDSS (m)")
axes[0].set_title("Well Vp in overlap")
axes[0].grid(True, alpha=0.25)

axes[1].plot(seis_amp, seis_depth, lw=0.8, color="tab:gray")
axes[1].axhspan(overlap_z_min, overlap_z_max, color="tab:green", alpha=0.15)
axes[1].invert_yaxis()
axes[1].set_xlabel("Amplitude")
axes[1].set_title("Depth-domain well trace")
axes[1].grid(True, alpha=0.25)

axes[2].plot(tdt_df["twt_s"] * 1000.0, tdt_df["tvdss_m"], lw=1.1)
axes[2].invert_yaxis()
axes[2].set_xlabel("Relative TWT (ms)")
axes[2].set_title("Local time-depth table")
axes[2].grid(True, alpha=0.25)

fig_depth_path = output_dir / f"qc_01_auto_tie_depth_time_match_{well_name}.png"
savefig(fig_depth_path)
plt.show()

In [ ]:
fig, ax = outputs.plot_optimization_objective(figsize=(6, 3))
fig_objective_path = output_dir / f"qc_02_auto_tie_optimization_objective_{well_name}.png"
savefig(fig_objective_path)
plt.show()


In [ ]:
_scale = 120000
fig, axes = outputs.plot_tie_window(wiggle_scale=_scale, figsize=(12.0, 7.5))
fig_tie_window_path = output_dir / f"qc_03a_auto_tie_window_raw_{well_name}.png"
savefig(fig_tie_window_path)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
t_ms = outputs.seismic.basis * 1000.0

axes[0].plot(outputs.r.values, t_ms, lw=0.8, color="tab:purple")
axes[0].invert_yaxis()
axes[0].set_xlabel("Reflectivity")
axes[0].set_ylabel("Relative TWT (ms)")
axes[0].set_title("Reflectivity")
axes[0].grid(True, alpha=0.25)

axes[1].plot(seis_norm, t_ms, lw=0.9, label="Seismic", color="black")
axes[1].plot(cropped_synthetic, t_ms, lw=0.9, label="Synthetic", color="tab:red", alpha=0.85)
axes[1].set_xlabel("Normalized amplitude")
axes[1].set_title(f"Auto-tie cropped: corr={cropped_corr:.3f}, nmae={cropped_nmae:.3f}")
axes[1].legend(loc="best")
axes[1].grid(True, alpha=0.25)

axes[2].plot(seis_norm - cropped_synthetic, t_ms, lw=0.9, color="tab:gray")
axes[2].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[2].set_xlabel("Residual")
axes[2].set_title("Residual")
axes[2].grid(True, alpha=0.25)

fig_synthetic_overlay_path = output_dir / f"qc_03b_auto_tie_cropped_synthetic_vs_seismic_{well_name}.png"
savefig(fig_synthetic_overlay_path)
plt.show()


In [ ]:
best_parameters, best_values = outputs.ax_client.get_best_parameters()  # type: ignore
best_objective_means, best_objective_covariances = best_values  # type: ignore

print("Auto-tie best parameters:")
for key in ["logs_median_size", "logs_median_threshold", "logs_std", "table_t_shift"]:
    value = best_parameters.get(key)
    if key == "table_t_shift" and value is not None:
        print(f"{key}: {value:.6f} s ({value * 1000.0:.2f} ms)")  # type: ignore
    else:
        print(f"{key}: {value}")

print("Objective means:", best_objective_means)
print("Objective covariances:", best_objective_covariances)


In [ ]:
fig, ax = viz.plot_td_table(inputs.table, plot_params=dict(label="initial"))
viz.plot_td_table(outputs.table, plot_params=dict(label="optimized"), fig_axes=(fig, ax))
ax.legend(loc="best")
fig_td_table_path = output_dir / f"qc_04_auto_tie_td_table_{well_name}.png"
savefig(fig_td_table_path)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(
    raw_wavelet.basis * 1000.0, raw_wavelet.values / np.max(np.abs(raw_wavelet.values)), lw=1.1, label="raw auto-tie"
)
axes[0].plot(
    cropped_wavelet.basis * 1000.0,
    cropped_wavelet.values / np.max(np.abs(cropped_wavelet.values)),
    lw=1.2,
    label="cropped 201 ms + energy norm",
)
axes[0].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[0].set_xlabel("Time (ms)")
axes[0].set_title("Wavelet time domain")
axes[0].legend(loc="best")
axes[0].grid(True, alpha=0.25)

freq_raw = np.fft.rfftfreq(raw_wavelet.size, d=raw_wavelet.sampling_rate)  # type: ignore
spec_raw = np.abs(np.fft.rfft(raw_wavelet.values))
freq_crop = np.fft.rfftfreq(cropped_wavelet.size, d=cropped_wavelet.sampling_rate)
spec_crop = np.abs(np.fft.rfft(cropped_wavelet.values))
axes[1].plot(freq_raw, spec_raw / np.max(spec_raw), lw=1.1, label="raw auto-tie")
axes[1].plot(freq_crop, spec_crop / np.max(spec_crop), lw=1.2, label="cropped 201 ms")
axes[1].set_xlim(0, min(125, freq_raw[-1]))
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_title("Normalized amplitude spectrum")
axes[1].legend(loc="best")
axes[1].grid(True, alpha=0.25)

fig_wavelet_path = output_dir / f"qc_05_auto_tie_wavelet_raw_vs_cropped_{well_name}.png"
savefig(fig_wavelet_path)
plt.show()


## 输出清单


In [ ]:
artifact_paths = [
    local_tdt_path,
    seismic_twt_path,
    raw_wavelet_path,
    cropped_wavelet_path,
    raw_tie_path,
    cropped_qc_path,
    summary_path,
    fig_depth_path,
    fig_objective_path,
    fig_tie_window_path,
    fig_synthetic_overlay_path,
    fig_td_table_path,
    fig_wavelet_path,
]
for path in artifact_paths:
    print(path)
